# Incremental Claims Parsing - Lakeflow Spark Declarative Pipeline (SDP)
### Workers' comp claims OCR -> structured, searchable data on Databricks

This is the **source for a Lakeflow Spark Declarative Pipeline (SDP)**, the successor to DLT. It defines streaming tables declaratively and the pipeline runtime handles **checkpointing, retries, and incremental processing automatically** (no manual `writeStream` or checkpoint paths).

Flow: `scanned PDFs in a Volume` -> **Auto Loader** streaming table (Bronze) -> `ai_parse_document` streaming table (Bronze, parsed once per doc) -> explode elements (Silver) -> `ai_extract` for a few claim fields (Silver+).

> **Important:** this notebook is pipeline source. Do **not** run it cell by cell. Attach it to a Lakeflow pipeline and click Start (see the last cell). The `dp` decorators only execute inside the pipeline runtime.

## Prerequisites

1. **AI Functions enabled** on the workspace (`ai_parse_document`, `ai_extract`, part of Agent Bricks). Confirm with your platform/security team it is approved for claims data (PII).
2. A **Unity Catalog Volume** with a sample of scanned claim PDFs (a few hundred to start, including hard scans: old, faxed, handwritten).
3. **DBR 15.4 LTS or above** and a Foundation Model APIs supported region. AI functions in pipelines require the **`preview` channel** (set per-table below).
4. The pipeline's **target catalog and schema** are set in the pipeline settings, not in this code. The `@dp.table` names below become tables in that schema.

## Configuration
Set the source Volume path. (Target catalog/schema are configured on the pipeline itself.)

In [ ]:
from pyspark import pipelines as dp
from pyspark.sql.functions import expr, current_timestamp

# Source Volume holding the scanned document PDFs.
# Set it on the pipeline as a configuration key `source_volume_path`, or edit the default below.
SOURCE_VOLUME_PATH = spark.conf.get("source_volume_path", "/Volumes/main/document_intelligence/raw_pdfs")

## Bronze 1 - ingest raw PDFs incrementally (Auto Loader)
A streaming table backed by Auto Loader (`cloudFiles`). It discovers and ingests only new files on each run. SDP manages the schema and checkpoint for you.

In [ ]:
@dp.table(
    name="claims_raw",
    comment="Raw scanned claim PDFs, ingested incrementally with Auto Loader.",
)
def claims_raw():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "binaryFile")
        .option("pathGlobFilter", "*.{pdf,PDF,jpg,jpeg,png,tif,tiff}")
        .option("recursiveFileLookup", "true")
        .load(SOURCE_VOLUME_PATH)
    )

## Bronze 2 - parse each document once with `ai_parse_document`
Because this is a **streaming table** reading `STREAM(claims_raw)`, every document is parsed exactly once. No reprocessing, no manual checkpoint. `pipelines.channel = preview` is required to use AI functions inside a pipeline.

In [ ]:
@dp.table(
    name="claims_parsed_bronze",
    comment="Parsed claims (OCR + layout) via ai_parse_document v2.0.",
    table_properties={"pipelines.channel": "preview"},
)
def claims_parsed_bronze():
    return (
        spark.readStream.table("claims_raw")
        .withColumn("parsed", expr("ai_parse_document(content, map('version', '2.0'))"))
        .withColumn("parsed_at", current_timestamp())
        .select("path", "length", "modificationTime", "parsed_at", "parsed")
    )

## Silver - explode parsed elements (one row per page element)
Explode is a stateless transform, so it runs fine on a streaming table. Using `spark.sql` here keeps the VARIANT navigation (`parsed:document.elements`) and `LATERAL VIEW EXPLODE` clean. The `error_status` field on the Bronze parsed output lets you isolate any documents that failed (often encrypted PDFs) and retry them separately.

In [ ]:
@dp.table(
    name="claims_elements_silver",
    comment="One row per parsed element (paragraph, table, etc.).",
)
def claims_elements_silver():
    # ai_parse_document returns a VARIANT, so cast the elements array to ARRAY<VARIANT>
    # before exploding, then read each field with variant accessors.
    return spark.sql("""
        SELECT
            path,
            element:page_id::int    AS page_id,
            element:id::int         AS element_id,
            element:type::string    AS element_type,
            element:content::string AS content,
            parsed_at
        FROM STREAM claims_parsed_bronze
        LATERAL VIEW EXPLODE(CAST(parsed:document.elements AS ARRAY<VARIANT>)) AS element
    """)

## Silver+ (optional) - extract a few claim fields with `ai_extract` v2
A teaser of structured extraction. Note the V2 call shape: schema as a JSON string plus `map('version','2.0')`. Swap in the fields that matter for your claims.

In [ ]:
@dp.table(
    name="claims_fields_silver",
    comment="Example structured field extraction via ai_extract v2.",
    table_properties={"pipelines.channel": "preview"},
)
def claims_fields_silver():
    # ai_extract v2 schema: each field is an object with a type (and optional description).
    return spark.sql("""
        SELECT
            path,
            ai_extract(
                content,
                '{"claim_number": {"type": "string", "description": "Workers comp claim number"}, "claimant_name": {"type": "string", "description": "Injured employee full name"}, "date_of_injury": {"type": "string", "description": "Date of injury"}, "employer": {"type": "string", "description": "Employer name"}, "body_part": {"type": "string", "description": "Injured body part or nature of injury"}}',
                map('version', '2.0')
            ) AS fields
        FROM STREAM claims_elements_silver
        WHERE element_type IN ('text', 'table') AND length(content) > 40
    """)

## Running this as a pipeline

1. **Create the pipeline**: Workflows -> Lakeflow Pipelines -> Create pipeline. Add this notebook as the source. Set a **target catalog and schema** for the output tables, and pick **Serverless**.
2. **Start it**: click Start. SDP builds `claims_raw` -> `claims_parsed_bronze` -> `claims_elements_silver` -> `claims_fields_silver` and tracks each file in a managed checkpoint, so re-runs only process new documents.
3. **Recovery is automatic**: if a run fails partway through millions of files, just trigger it again. The managed checkpoints skip everything already parsed. (Avoid renaming the tables/flows later, since flow names key the checkpoints.)
4. **Throughput tips**:
   - Performance scales with **pages, not files**. A few very long documents can dominate a run.
   - For documents over 500 pages or 100 MB, parse in ranges: `ai_parse_document(content, map('version','2.0','pageRange','1-100'))`.
   - At sustained millions scale, the AI Functions backend may rate limit. If you see throttling, request a model-unit / provisioned-throughput increase.
   - Documents that fail (often encrypted PDFs) show up via `parsed:error_status`. Re-render those to images and re-run, or handle them in a small side flow.

Once the Silver tables are populated, point Vector Search (or AI/BI Genie) at them for smart search over the full claims history.